# 01 - Data Pipeline

Builds the raw dataset for a **free-data reproduction** of the safe-haven
regression (Ranaldo & Söderlind, 2010; thesis Table 1). The original study draws
on Bloomberg and Refinitiv; here every series is replaced by a free equivalent ,
FRED for macro/FX/rates and Yahoo Finance for the equity index, covering
**2000-01-01 to 2024-06-30**. Output: one tidy table in `../data/`.

| Thesis series | Free substitute | Note |
|---|---|---|
| EUR/GBP/JPY/CHF vs USD (Bloomberg) | FRED `DEXUSEU`, `DEXUSUK`, `DEXJPUS`, `DEXSZUS` | JPY, CHF reciprocated to USD/unit |
| S&P 500 (Bloomberg) | Yahoo Finance `^GSPC` | FRED index limited to 10y by licence |
| VIX (Bloomberg) | FRED `VIXCLS` | |
| 10Y UST future (Bloomberg) | FRED `DGS10` (yield) | yield, not futures price, see 02 |
| US 3M T-Bill (FRED) | FRED `DTB3` | already free in the thesis |
| 3M interbank US/EUR/GBP/JPY/CHF (Bloomberg/Refinitiv) | FRED OECD `IR3TIB01*M156N` | monthly, forward-filled to daily |

The notebook runs in three steps: **Setup**, **Load**, **Persist**.

## 1. Setup

Imports, configuration (sample window and source maps), and the FRED client.
`FX_SERIES` records which quotes are reciprocated so every rate reads as **USD
per unit of foreign currency**; `INTERBANK` are OECD monthly 3-month rates.

In [1]:
import os
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import yfinance as yf
from dotenv import load_dotenv
from fredapi import Fred

START, END = "2000-01-01", "2024-06-30"
DATA_DIR = Path("..") / "data"

FX_SERIES = {
    "DEXUSEU": ("EUR", False),
    "DEXUSUK": ("GBP", False),
    "DEXJPUS": ("JPY", True),
    "DEXSZUS": ("CHF", True),
}
MACRO_SERIES = {
    "VIXCLS": "VIX",
    "DGS10": "UST10Y",     # 10-year Treasury yield (% p.a.)
    "DTB3": "TBILL_3M",    # 3-month Treasury bill (% p.a.)
}
INTERBANK = {              # OECD 3-month interbank rates (monthly, % p.a.)
    "IR3TIB01USM156N": "US_3M",
    "IR3TIB01EZM156N": "EUR_3M",
    "IR3TIB01GBM156N": "GBP_3M",
    "IR3TIB01JPM156N": "JPY_3M",
    "IR3TIB01CHM156N": "CHF_3M",
}

load_dotenv(Path("..") / ".env")
if "FRED_API_KEY" not in os.environ:
    raise RuntimeError(
        "FRED_API_KEY missing - this notebook only re-downloads the raw data. "
        "The committed data/levels.csv already covers the full sample, so "
        "notebooks 02-09 run without it; add the key to ../.env only to "
        "refresh the data.")
fred = Fred(api_key=os.environ["FRED_API_KEY"])

def fred_series(code):
    return fred.get_series(code, observation_start=START, observation_end=END)

## 2. Load

Fetch every series from its free source and assemble one daily table: FX and
macro from FRED, the equity index from Yahoo Finance, and the monthly interbank
rates forward-filled onto the daily index.

In [2]:
# Exchange rates (reciprocate JPY, CHF) and macro indicators - FRED, daily
fx = {}
for code, (label, invert) in FX_SERIES.items():
    s = fred_series(code)
    fx[label] = 1 / s if invert else s
fx = pd.DataFrame(fx)
macro = pd.DataFrame({label: fred_series(code) for code, label in MACRO_SERIES.items()})

# Equity index - Yahoo Finance (full history), aligned tz-naive to FRED dates
sp = yf.download("^GSPC", start=START, end=END, auto_adjust=True, progress=False)["Close"]
if isinstance(sp, pd.DataFrame):
    sp = sp.iloc[:, 0]
_idx = pd.DatetimeIndex(sp.index)
sp.index = _idx.tz_localize(None) if _idx.tz is not None else _idx
sp = sp.rename("SP500")

# Interbank rates - OECD via FRED, monthly
interbank_m = pd.DataFrame({label: fred_series(code) for code, label in INTERBANK.items()})

# Assemble: daily block, then forward-fill monthly interbank onto the daily index
data = pd.concat([fx, macro, sp], axis=1).sort_index().loc[START:END]
data = pd.concat([data, interbank_m.reindex(data.index, method="ffill")], axis=1)
data = data.rename_axis("date")

print("Assembled:", data.shape, "(rows, cols)")
data.describe().round(3)

Assembled: (6390, 13) (rows, cols)


,EUR,GBP,JPY,CHF,VIX,UST10Y,TBILL_3M,SP500,US_3M,EUR_3M,GBP_3M,JPY_3M,CHF_3M
count,6142.000,6142.000,6142.000,6142.000,6178.000,6127.000,6127.000,6161.000,6368.000,6390.000,6390.000,5805.000,6390.000
mean,1.192,1.532,0.009,0.938,19.922,3.266,1.744,2036.946,2.060,1.570,2.544,0.208,0.435
std,0.158,0.223,0.001,0.168,8.521,1.313,1.894,1140.189,1.999,1.804,2.222,0.246,1.180
min,0.827,1.070,0.006,0.548,9.140,0.520,-0.050,676.530,0.090,-0.582,0.030,-0.072,-0.930
25%,1.090,1.330,0.008,0.810,13.840,2.190,0.100,1196.540,0.280,-0.249,0.570,0.056,-0.706
50%,1.184,1.529,0.009,1.005,17.880,3.210,1.060,1471.560,1.290,1.048,1.005,0.098,0.099
75%,1.309,1.647,0.010,1.070,23.490,4.275,2.910,2681.660,3.480,3.310,4.735,0.332,1.302
max,1.601,2.110,0.013,1.371,82.690,6.790,6.240,5487.030,6.730,5.113,6.647,0.890,3.347


## 3. Persist

Write the snapshot and a data README (with source table and download date) so
the results reproduce even if a source later revises or discontinues a series.

In [3]:
DATA_DIR.mkdir(exist_ok=True)
downloaded = date.today().isoformat()

levels_path = DATA_DIR / "levels.csv"
data.to_csv(levels_path)

readme = f"""# Data

`levels.csv` holds daily levels from {START} to {END} - a free-data reproduction
of the safe-haven dataset (originally Bloomberg / Refinitiv).

## Columns

| Column   | Description                        | Source                      |
| -------- | ---------------------------------- | --------------------------- |
| EUR      | USD per euro                       | FRED DEXUSEU                |
| GBP      | USD per pound                      | FRED DEXUSUK                |
| JPY      | USD per yen                        | FRED DEXJPUS (reciprocated) |
| CHF      | USD per franc                      | FRED DEXSZUS (reciprocated) |
| SP500    | S&P 500 index level                | Yahoo Finance ^GSPC         |
| VIX      | CBOE volatility index              | FRED VIXCLS                 |
| UST10Y   | 10-year Treasury yield (% p.a.)    | FRED DGS10                  |
| TBILL_3M | 3-month Treasury bill (% p.a.)     | FRED DTB3                   |
| US_3M    | US 3-month interbank rate (% p.a.) | FRED IR3TIB01USM156N        |
| EUR_3M   | Euro-area 3-month interbank rate   | FRED IR3TIB01EZM156N        |
| GBP_3M   | UK 3-month interbank rate          | FRED IR3TIB01GBM156N        |
| JPY_3M   | Japan 3-month interbank rate       | FRED IR3TIB01JPM156N        |
| CHF_3M   | Switzerland 3-month interbank rate | FRED IR3TIB01CHM156N        |

## Notes

- Exchange rates are quoted as USD per unit of foreign currency; JPY and CHF are
  reciprocated from their native FRED quotes.
- Interbank rates are OECD monthly series, forward-filled to daily frequency.
- The 10-year term uses the Treasury yield (DGS10), not the futures price used in
  the thesis; the notebooks use the daily change of this yield (percentage points)
  as the regressor.
- Japan's interbank series starts in 2002, so JPY_3M has fewer observations.

_Downloaded: {downloaded}_
"""
(DATA_DIR / "README.md").write_text(readme)

print(f"Saved {data.shape[0]} rows x {data.shape[1]} cols -> {levels_path}")
print(f"Wrote {DATA_DIR / 'README.md'} (downloaded {downloaded})")

Saved 6390 rows x 13 cols -> ../data/levels.csv
Wrote ../data/README.md (downloaded 2026-07-19)
